# Cloud ML Deployment — Concepts, Options, and Costs

Running a model on your laptop is useful during development. Deploying it to the cloud means real users can call it 24/7, from anywhere, at any scale — without you managing physical servers.

This notebook covers:
- What cloud deployment adds over local serving
- The three main deployment options and when to pick each
- The full deployment lifecycle
- A simple cost model for estimating cloud spend

## Learning Objectives

By the end of this notebook you will be able to:
1. Name three cloud deployment options and explain the trade-offs between them
2. Train, save, and package a model artifact for cloud deployment
3. Simulate a cloud storage upload step and explain what it replaces in production
4. Estimate the cost difference between on-demand and spot pricing for a given request volume

## 1. What Cloud Deployment Adds

| Capability | Local server | Cloud managed endpoint |
|---|---|---|
| Scaling to traffic spikes | Manual — restart with more workers | Automatic horizontal scaling |
| Global availability | Single region | Multi-region, CDN-backed |
| SLA / uptime guarantee | None | 99.9 – 99.99% |
| Infrastructure patching | You do it | Provider does it |
| Billing | Fixed server cost | Pay per request / per hour |

The cost of managed services is higher per compute unit, but the operational overhead is dramatically lower.

## 2. The Three Deployment Options

**Option A — Raw VM (EC2 / Compute Engine / Azure VM)**
- You rent a virtual machine, install Python, run your FastAPI/Flask server yourself
- Full control, but you manage OS updates, scaling, load balancers, and monitoring
- Best for: custom hardware needs, legacy constraints, very predictable traffic

**Option B — Managed containers (ECS / GKE / AKS)**
- You build a Docker image, push it to a registry, define scaling rules
- The cloud orchestrator manages where and how many containers run
- Best for: teams already using Docker, need for flexibility without full server ops

**Option C — Managed ML endpoints (SageMaker / Vertex AI / Azure ML)**
- You upload a model artifact + inference script; the platform handles everything else
- Built-in A/B testing, model monitoring, auto-scaling, integrated billing
- Best for: pure ML teams who want to ship fast without DevOps expertise

## 3. Train and Save a Model Artifact

In [1]:
import numpy as np
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load data
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale and train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_s, y_train)

acc = accuracy_score(y_test, clf.predict(X_test_s))
print(f"Test accuracy: {acc:.2%}")

# Save model + scaler together as a pipeline artifact
artifact = {'model': clf, 'scaler': scaler, 'classes': iris.target_names.tolist()}
joblib.dump(artifact, '/tmp/iris_rf_artifact.joblib')
print("Artifact saved: /tmp/iris_rf_artifact.joblib")

<frozen importlib._bootstrap>:219: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


Test accuracy: 100.00%
Artifact saved: /tmp/iris_rf_artifact.joblib


## 4. Simulate a Cloud Storage Upload

In production the model artifact goes to object storage first:
- **AWS**: S3 bucket (`s3://my-bucket/models/iris/v1/`)
- **GCP**: Google Cloud Storage (`gs://my-bucket/models/iris/v1/`)
- **Azure**: Blob Storage (`https://account.blob.core.windows.net/models/`)

The managed ML service then pulls the artifact from there. We simulate this with a local temp directory.

In [2]:
import shutil
import pathlib
import hashlib

# Simulate cloud storage as a local directory
SIMULATED_BUCKET = pathlib.Path('/tmp/simulated_cloud_storage/models/iris/v1')
SIMULATED_BUCKET.mkdir(parents=True, exist_ok=True)

def simulate_upload(local_path, destination_bucket):
    """Mirrors what boto3 s3.upload_file() or gsutil cp would do."""
    src = pathlib.Path(local_path)
    dst = destination_bucket / src.name
    shutil.copy2(src, dst)

    # Compute checksum to verify integrity (same as S3 ETag)
    md5 = hashlib.md5(dst.read_bytes()).hexdigest()
    size_kb = dst.stat().st_size / 1024
    print(f"Uploaded: {dst}")
    print(f"  Size  : {size_kb:.1f} KB")
    print(f"  MD5   : {md5}")
    return str(dst)

artifact_path = simulate_upload('/tmp/iris_rf_artifact.joblib', SIMULATED_BUCKET)

print("\nIn production this would be:")
print("  import boto3")
print("  s3 = boto3.client('s3')")
print("  s3.upload_file('iris_rf_artifact.joblib', 'my-bucket', 'models/iris/v1/iris_rf_artifact.joblib')")

Uploaded: /tmp/simulated_cloud_storage/models/iris/v1/iris_rf_artifact.joblib
  Size  : 183.0 KB
  MD5   : 874cfb481be59f6a01a2a435b5e2af66

In production this would be:
  import boto3
  s3 = boto3.client('s3')
  s3.upload_file('iris_rf_artifact.joblib', 'my-bucket', 'models/iris/v1/iris_rf_artifact.joblib')


## 5. Cost Model — On-Demand vs Spot Pricing

Cloud providers offer two pricing modes:
- **On-demand**: fixed hourly rate, instance available whenever you need it
- **Spot / Preemptible**: up to 90% cheaper, but the instance can be terminated with 2 minutes notice

For inference endpoints that must always be available, on-demand (or reserved) instances are required. Spot is great for batch jobs.

In [3]:
# Cost model for serving 1 million predictions per day

# Instance specs (ml.m5.large equivalent)
ON_DEMAND_PER_HOUR = 0.115   # USD
SPOT_PER_HOUR      = 0.034   # USD (~70% discount)

# Throughput assumptions
REQUESTS_PER_DAY   = 1_000_000
REQUESTS_PER_HOUR  = REQUESTS_PER_DAY / 24
REQUESTS_PER_SECOND = REQUESTS_PER_HOUR / 3600

# One instance handles ~50 req/s for a simple sklearn model
CAPACITY_PER_INSTANCE = 50
INSTANCES_NEEDED = max(1, int(REQUESTS_PER_SECOND / CAPACITY_PER_INSTANCE) + 1)

hours_per_month = 24 * 30
on_demand_monthly = INSTANCES_NEEDED * ON_DEMAND_PER_HOUR * hours_per_month
spot_monthly = INSTANCES_NEEDED * SPOT_PER_HOUR * hours_per_month

cost_per_million_on_demand = (on_demand_monthly / (REQUESTS_PER_DAY * 30)) * 1_000_000
cost_per_million_spot = (spot_monthly / (REQUESTS_PER_DAY * 30)) * 1_000_000

print(f"Daily request volume  : {REQUESTS_PER_DAY:,}")
print(f"Peak req/s            : {REQUESTS_PER_SECOND:.1f}")
print(f"Instances needed      : {INSTANCES_NEEDED}")
print()
print(f"On-demand monthly cost: ${on_demand_monthly:,.2f}")
print(f"Spot monthly cost     : ${spot_monthly:,.2f}")
print(f"Savings with spot     : ${on_demand_monthly - spot_monthly:,.2f} ({(1-spot_monthly/on_demand_monthly)*100:.0f}%)")
print()
print(f"Cost per 1M predictions (on-demand): ${cost_per_million_on_demand:.4f}")
print(f"Cost per 1M predictions (spot)     : ${cost_per_million_spot:.4f}")
print()
print("WARNING: Spot instances can be terminated with 2 min notice.")
print("Never use spot-only for always-on inference endpoints.")

Daily request volume  : 1,000,000
Peak req/s            : 11.6
Instances needed      : 1

On-demand monthly cost: $82.80
Spot monthly cost     : $24.48
Savings with spot     : $58.32 (70%)

Cost per 1M predictions (on-demand): $2.7600
Cost per 1M predictions (spot)     : $0.8160

Never use spot-only for always-on inference endpoints.


## 6. Deployment Lifecycle Diagram

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

stages = [
    ('Train', '#4C72B0'),
    ('Package\n(tar.gz / joblib)', '#DD8452'),
    ('Push to\nRegistry / Storage', '#55A868'),
    ('Deploy\nEndpoint', '#C44E52'),
    ('Monitor\n& Retrain', '#8172B2'),
]

fig, ax = plt.subplots(figsize=(12, 3))
ax.set_xlim(0, len(stages))
ax.set_ylim(0, 1)
ax.axis('off')

for i, (label, color) in enumerate(stages):
    x = i + 0.5
    box = mpatches.FancyBboxPatch(
        (i + 0.05, 0.2), 0.9, 0.6,
        boxstyle='round,pad=0.05',
        facecolor=color, edgecolor='white', linewidth=2
    )
    ax.add_patch(box)
    ax.text(x, 0.5, label, ha='center', va='center',
            color='white', fontsize=9, fontweight='bold')
    if i < len(stages) - 1:
        ax.annotate('', xy=(i + 1.05, 0.5), xytext=(i + 0.95, 0.5),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_title('ML Deployment Lifecycle', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('/tmp/deployment_lifecycle.png', dpi=100, bbox_inches='tight')
plt.show()
print("Lifecycle diagram saved to /tmp/deployment_lifecycle.png")

Lifecycle diagram saved to /tmp/deployment_lifecycle.png


/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_25274/3431907879.py:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 7. When to Choose Each Option

| Situation | Best choice |
|---|---|
| Small team, want to ship fast | Managed ML endpoint (SageMaker / Vertex / Azure ML) |
| Already have Docker expertise | Managed containers (ECS / GKE / AKS) |
| Custom GPU setup, non-standard OS | Raw VM (EC2 / Compute Engine) |
| Offline batch scoring (not real-time) | Batch transform job — no endpoint needed |
| Experimental / low traffic | Cloud Run / Lambda (serverless) — scales to zero |

For most new ML projects, start with a managed ML endpoint and move down only when you have a specific reason.

## Summary

- Cloud deployment adds **managed scaling, global availability, SLAs, and infrastructure management** over a local server
- The three options are: raw VM (most control), managed containers (middle ground), managed ML endpoints (least ops overhead)
- The deployment lifecycle is: **Train → Package → Push to Storage → Deploy Endpoint → Monitor**
- Spot/preemptible instances save up to 90% but can be terminated at any time — never use them alone for always-on endpoints
- The model artifact (joblib / tar.gz) goes to cloud object storage first; the ML platform pulls from there

## Self-Check

Answer these before moving to the next notebook:

1. **What does a managed ML endpoint give you that a raw VM does not?**
   *(Think about what you no longer have to configure or maintain.)*

2. **Why is spot/preemptible pricing risky for an always-on inference endpoint?**
   *(What can happen to your instance, and when?)*

3. **What is the difference between deploying a model and deploying a service?**
   *(A model is a file. A service is... what exactly?)*